In [9]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
# Đọc dữ liệu
df = pd.read_csv('./products.csv', nrows=50000)
df.head(3)

,Product_ID,name,main_category,sub_category,ratings,no_of_ratings,PRICE
0,1,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...",Electronics,Electronic Gadgets,4.0,965,AUD 358.47
1,2,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...",Electronics,Electronic Gadgets,4.3,226,AUD 377.34
2,3,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,Electronics,Electronic Gadgets,4.2,54,AUD 43.38


In [11]:
import re
# Function to clean the product names
def clean_product_name(name):
    name = re.sub(r'[\(\)\|,:"&\-]', '', name)  # remove special characters 
    name = re.sub(r'\s+', ' ', name)  
    return name.strip()  

df['cleaned_name'] = df['name'].apply(clean_product_name)
df[['name', 'cleaned_name']].head(10)

,name,cleaned_name
0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...",Redmi 10 Power Power Black 8GB RAM 128GB Storage
1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...",OnePlus Nord CE 2 Lite 5G Blue Tide 6GB RAM 12...
2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...
3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...",Samsung Galaxy M33 5G Mystique Green 6GB 128GB...
4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...",OnePlus Nord CE 2 Lite 5G Black Dusk 6GB RAM 1...
5,"Redmi 10 Power (Sporty Orange, 8GB RAM, 128GB ...",Redmi 10 Power Sporty Orange 8GB RAM 128GB Sto...
6,boAt Airdopes 141 Bluetooth Truly Wireless in ...,boAt Airdopes 141 Bluetooth Truly Wireless in ...
7,"Apple 20W USB-C Power Adapter (for iPhone, iPa...",Apple 20W USBC Power Adapter for iPhone iPad A...
8,"Fire-Boltt Ninja Call Pro Plus 1.83"" Smart Wat...",FireBoltt Ninja Call Pro Plus 1.83 Smart Watch...
9,"Samsung Galaxy M33 5G (Emerald Brown, 6GB, 128...",Samsung Galaxy M33 5G Emerald Brown 6GB 128GB ...


In [12]:
df['product_details'] = (
    df['name'] + ' ' + 
    df['main_category'] + ' ' + 
    df['sub_category'] 
)

In [13]:
tfidf_vec = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2, ngram_range=(1, 1))
tfidf_matrix = tfidf_vec.fit_transform(df['product_details'])

In [14]:
n_topics = 10  
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_matrix = lda.fit_transform(tfidf_matrix)

In [15]:
similarity_matrix = cosine_similarity(lda_matrix)

In [16]:
def plsa_based_recommendations(product_index, similarity_matrix, data, top_n=10):
    similar_indices = similarity_matrix[product_index].argsort()[::-1]
    similar_indices = similar_indices[similar_indices != product_index]
    top_indices = similar_indices[:top_n]
    return data.iloc[top_indices], similarity_matrix[product_index][top_indices]


In [25]:
try:
    input_product_index = int(input("Product Index: "))
    if input_product_index < 0 or input_product_index >= len(df):
        raise ValueError("Chỉ số sai.")
except ValueError as e:
    print(f"Lỗi: {e}")
    exit()

recommendations_result, similarities = plsa_based_recommendations(input_product_index, similarity_matrix, df, top_n=10)

# Hiển thị kết quả
print("\nĐề xuất:")
print(recommendations_result[['product_details']])


Đề xuất:
                                         product_details
12     OnePlus Nord CE 2 Lite 5G (Blue Tide, 8GB RAM,...
3415   Amazon Basics USB Type C to USB 3.1 Gen1 Type-...
7290   Car Charger, 120W Car Cigarette Lighter Splitt...
167    pTron Solero MB301 3A Micro USB Data & Chargin...
1352   Cable Management Box With Mobile Stand - Cord ...
7570   Google Pixel 6a 5G (Chalk, 6GB RAM, 128GB Stor...
19096  ILT Retail -(Combo Set of 5) Wired Durable, St...
3189   Sounce VGA to HDMI 1080P Full HD Mini VGA to H...
17217  Zebion ZEB-1002 Wired Earphone Oval Shape with...
4528   AMKETTE PowerPro iPhone Fast Charging Cable, U...
